In [1]:
import gymnasium as gym
import numpy as np
import random

In [2]:
# Sample Environment
env = gym.make("CliffWalking-v1")

In [3]:
print("States:", env.observation_space.n) # No. of states, or boxes in the grid
print("Actions:", env.action_space.n) # No. of actions, like up, down, left, right
start_state,_ = (env.reset()) # Start state = 3*12 + 0, i.e. row*12 + col, because rows = 4, cols = 12
print("Starting State after resetting the Environment:", start_state)

States: 48
Actions: 4
Starting State after resetting the Environment: 36


## Q-Learning

In [4]:
alpha = 0.5
gamma = 0.99
epsilon = 0.1
episodes = 500 

### Policy - Epsilon Greedy Policy

In [5]:
# Q-table => stores Q values (Action Values)
Q = np.zeros((48,4)) # Size = 48 (Total States) X 4 (Possible Actions)

In [6]:
def epsilon_greedy(state):
    if random.random() < epsilon: # Exploration
        return env.action_space.sample()
    else: # Exploitation
        return np.argmax(Q[state])

### Training

In [7]:
rewards = []
episode_lens = []

for episode in range(episodes):
    # Option 1
    # render = (episode % 50 == 0)
    # if render:
    #     env = gym.make("CliffWalking-v1", render_mode="human")
    # else:
    #     env = gym.make("CliffWalking-v1")
        
    # Option 2
    env = gym.make("CliffWalking-v1")
    
    done = False
    state,_ = env.reset()
    total_reward = 0
    episode_len = 0

    while not done:
        action = epsilon_greedy(state)
        
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Q-Learn Update
        Q[state, action] += alpha * (reward + gamma*np.max(Q[next_state]) - Q[state, action]) # Off Policy
        state = next_state

        total_reward += reward
        episode_len += 1

    rewards.append(total_reward)
    episode_lens.append(episode_len)

    print(f"Episode: {episode + 1}/{episodes} => Reward: {total_reward} => Episode Length: {episode_len}")
    env.close()

Episode: 1/500 => Reward: -56 => Episode Length: 56
Episode: 2/500 => Reward: -1940 => Episode Length: 554
Episode: 3/500 => Reward: -225 => Episode Length: 126
Episode: 4/500 => Reward: -50 => Episode Length: 50
Episode: 5/500 => Reward: -191 => Episode Length: 92
Episode: 6/500 => Reward: -199 => Episode Length: 100
Episode: 7/500 => Reward: -56 => Episode Length: 56
Episode: 8/500 => Reward: -261 => Episode Length: 63
Episode: 9/500 => Reward: -192 => Episode Length: 93
Episode: 10/500 => Reward: -150 => Episode Length: 51
Episode: 11/500 => Reward: -151 => Episode Length: 52
Episode: 12/500 => Reward: -51 => Episode Length: 51
Episode: 13/500 => Reward: -71 => Episode Length: 71
Episode: 14/500 => Reward: -22 => Episode Length: 22
Episode: 15/500 => Reward: -66 => Episode Length: 66
Episode: 16/500 => Reward: -37 => Episode Length: 37
Episode: 17/500 => Reward: -75 => Episode Length: 75
Episode: 18/500 => Reward: -148 => Episode Length: 49
Episode: 19/500 => Reward: -30 => Episode 

In [8]:
print("Agent with max. reward")
idx_reward = np.argmax(rewards)
print(f"Episode: {idx_reward + 1}")
print(f"Reward: {rewards[idx_reward]}")
print(f"Episode Length: {episode_lens[idx_reward]}")
print("-"*95)
print("Agent with min. episode length")
idx_episode_len = np.argmin(episode_lens)
print(f"Episode: {idx_episode_len + 1}")
print(f"Reward: {rewards[idx_episode_len]}")
print(f"Episode Length: {episode_lens[idx_episode_len]}")

Agent with max. reward
Episode: 51
Reward: -13
Episode Length: 13
-----------------------------------------------------------------------------------------------
Agent with min. episode length
Episode: 51
Reward: -13
Episode Length: 13


### What did our Agent learn ?

In [9]:
env = gym.make("CliffWalking-v1", render_mode="human")
state,_ = env.reset()
total_reward = 0
episode_len = 0
done = False

while not done:
    action = np.argmax(Q[state])
    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

    episode_len += 1
    total_reward += reward

print(f"Reward: {total_reward} => Episode Length: {episode_len}")
env.close()

C:\Users\heer4\anaconda3\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Reward: -13 => Episode Length: 13


In [10]:
# By default, in evry row, the idx in the Q-table means:
# 0: Move up
# 1: Move right
# 2: Move down
# 3: Move left

# FInd the direction to be taken on the starting cell
Q[36] # It shows that at idx 0, reward is the highest, & hence it's better to move upwards
# For boundary cells, if the movement is such that the agent goes out of the grid, then the reward is same as the cell, it is standing on.

array([ -12.2478977 , -111.86012855,  -13.1254155 ,  -13.12540168])